# DX 704 Week 8 Project

This homework will modify a simulator controlling a small vehicle to implement tabular q-learning.
You will first test your code with random and greedy-epsilon policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [1]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction = (curr_direction + curr_action) % 8

        next_state = self.construct_state(next_x, next_y, next_direction)
        next_reward = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        return random.choice(self.initial_states)

In [2]:
simulator = RoverSimulator(16)
initial_sample = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 1922


## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [3]:
# YOUR CHANGES HERE

def random_policy(state, actions):
    return random.choice(actions)

Use the code below to test your random policy.
Then modify it to save the results in "log-random.tsv" with the columns curr_state, curr_action, next_reward and next_state.

In [4]:
# YOUR CHANGES HERE

for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
    print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)

CURR STATE 1924 ACTION 0 NEXT REWARD 0 NEXT STATE 1796
CURR STATE 1796 ACTION -1 NEXT REWARD 0 NEXT STATE 1667
CURR STATE 1667 ACTION -1 NEXT REWARD 0 NEXT STATE 1546
CURR STATE 1546 ACTION -1 NEXT REWARD 0 NEXT STATE 1553
CURR STATE 1553 ACTION 1 NEXT REWARD 0 NEXT STATE 1690
CURR STATE 1690 ACTION 1 NEXT REWARD 0 NEXT STATE 1699
CURR STATE 1699 ACTION 1 NEXT REWARD 0 NEXT STATE 1580
CURR STATE 1580 ACTION -1 NEXT REWARD 0 NEXT STATE 1451
CURR STATE 1451 ACTION -1 NEXT REWARD 0 NEXT STATE 1330
CURR STATE 1330 ACTION -1 NEXT REWARD 0 NEXT STATE 1337
CURR STATE 1337 ACTION -1 NEXT REWARD 0 NEXT STATE 1472
CURR STATE 1472 ACTION 1 NEXT REWARD 0 NEXT STATE 1601
CURR STATE 1601 ACTION -1 NEXT REWARD 0 NEXT STATE 1736
CURR STATE 1736 ACTION 1 NEXT REWARD 0 NEXT STATE 1865
CURR STATE 1865 ACTION -1 NEXT REWARD 0 NEXT STATE 2000
CURR STATE 2000 ACTION 1 NEXT REWARD 0 NEXT STATE 2001
CURR STATE 2001 ACTION 0 NEXT REWARD 0 NEXT STATE 2009
CURR STATE 2009 ACTION -1 NEXT REWARD 0 NEXT STATE 2016


In [5]:
#save results to tsv file
import pandas as pd
rows_p1 = [] #store in list
for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
    rows_p1.append({
        "curr_state": curr_state,
        "curr_action": curr_action,
        "next_reward": next_reward,
        "next_state": next_state
    })
df1 = pd.DataFrame(rows_p1)
df1.head(5)

,curr_state,curr_action,next_reward,next_state
0,1095,0,0,1215
1,1215,0,0,1335
2,1335,0,0,1455
3,1455,1,0,1568
4,1568,1,0,1697


In [6]:
df1.to_csv("log-random.tsv", sep='\t', index=False)

Submit "log-random.tsv" in Gradescope.

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [7]:
# YOUR CHANGES HERE

Q = {} #initialize q-table
rows_p2 = [] #store results 

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=1024):
        if curr_state not in Q:
            Q[curr_state] = {a: 0 for a in simulator.get_actions(curr_state)}
        if next_state not in Q:
            Q[next_state] = {a: 0 for a in simulator.get_actions(next_state)}
        
        #store old val.
        old_val = Q[curr_state][curr_action]

        #get max future Q val.
        max_next_Q = max(Q[next_state].values())
        new_val = next_reward + 0.9 * max_next_Q #Q learning update

        #update Q
        Q[curr_state][curr_action] = new_val

        #add new rows to list
        rows_p2.append({
            "curr_state": curr_state,
            "curr_action": curr_action,
            "next_reward": next_reward,
            "next_state": next_state,
            "old_value": old_val,
            "new_value": new_val
        })
        print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)
        #pass

CURR STATE 1031 ACTION -1 NEXT REWARD 0 NEXT STATE 1158
CURR STATE 1158 ACTION 0 NEXT REWARD 0 NEXT STATE 1158
CURR STATE 1158 ACTION 1 NEXT REWARD 0 NEXT STATE 1159
CURR STATE 1159 ACTION 1 NEXT REWARD 0 NEXT STATE 1280
CURR STATE 1280 ACTION -1 NEXT REWARD 0 NEXT STATE 1415
CURR STATE 1415 ACTION 0 NEXT REWARD 0 NEXT STATE 1543
CURR STATE 1543 ACTION 1 NEXT REWARD 0 NEXT STATE 1664
CURR STATE 1664 ACTION 1 NEXT REWARD 0 NEXT STATE 1793
CURR STATE 1793 ACTION -1 NEXT REWARD 0 NEXT STATE 1928
CURR STATE 1928 ACTION -1 NEXT REWARD 0 NEXT STATE 1935
CURR STATE 1935 ACTION 1 NEXT REWARD 0 NEXT STATE 1920
CURR STATE 1920 ACTION 1 NEXT REWARD 0 NEXT STATE 1921
CURR STATE 1921 ACTION -1 NEXT REWARD 0 NEXT STATE 1928
CURR STATE 1928 ACTION 0 NEXT REWARD 0 NEXT STATE 1928
CURR STATE 1928 ACTION 0 NEXT REWARD 0 NEXT STATE 1928
CURR STATE 1928 ACTION -1 NEXT REWARD 0 NEXT STATE 1935
CURR STATE 1935 ACTION 0 NEXT REWARD 0 NEXT STATE 1927
CURR STATE 1927 ACTION 0 NEXT REWARD 0 NEXT STATE 1927
CURR

Save each step in the simulator in a file "q-random.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

In [8]:
# YOUR CHANGES HERE
df2 = pd.DataFrame(rows_p2)
df2.head(5)

,curr_state,curr_action,next_reward,next_state,old_value,new_value
0,1031,-1,0,1158,0.0,0.0
1,1158,0,0,1158,0.0,0.0
2,1158,1,0,1159,0.0,0.0
3,1159,1,0,1280,0.0,0.0
4,1280,-1,0,1415,0.0,0.0


In [9]:
df2.to_csv("q-random.tsv", sep='\t', index=False)

Submit "q-random.tsv" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

In [10]:
# YOUR CHANGES HERE

# hard-code epsilon=0.25. this is high but the environment is deterministic.
epsilon = 0.25
def epsilon_greedy_policy(state, actions):
    #explore: 25% of time
    if random.random() < epsilon:
        return random.choice(actions)
    
    #exploit: 75% of time
    if state in Q:
        return max(Q[state], key=Q[state].get) #pick action w/highest Q
    else:
        return random.choice(actions)

Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "q-greedy.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [11]:
# YOUR CHANGES HERE

Q = {} #reset Q
rows_p3 = [] #store results

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
        #print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)
        if curr_state not in Q:
            Q[curr_state] = {a: 0 for a in simulator.get_actions(curr_state)}
        if next_state not in Q:
            Q[next_state] = {a: 0 for a in simulator.get_actions(next_state)}

        #store old val.
        old_val = Q[curr_state][curr_action]

        #compute new val
        max_next_Q = max(Q[next_state].values())
        new_val = next_reward + 0.9 * max_next_Q

        Q[curr_state][curr_action] = new_val

        #store results
        rows_p3.append({
            "curr_state": curr_state,
            "curr_action": curr_action,
            "next_reward": next_reward,
            "next_state": next_state,
            "old_value": old_val,
            "new_value": new_val
        })

#save results in df
df3 = pd.DataFrame(rows_p3)
df3.head(5)

,curr_state,curr_action,next_reward,next_state,old_value,new_value
0,1031,1,0,1152,0.0,0.0
1,1152,-1,0,1287,0.0,0.0
2,1287,-1,0,1414,0.0,0.0
3,1414,-1,0,1413,0.0,0.0
4,1413,-1,0,1284,0.0,0.0


In [12]:
df3.to_csv("q-greedy.tsv", sep='\t', index=False)

Submit "q-greedy.tsv" in Gradescope.

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "policy-greedy.tsv" with columns state and action.

In [14]:
# YOUR CHANGES HERE

rows_p4 = [] #store results
states = df3["curr_state"].unique() #extract states seen in q-greedy.tsv

for state in states:
    if state in Q:
        best_action = max(Q[state], key=Q[state].get)
    else:
        best_action = 0 #fallback

    rows_p4.append({
        "state": state,
        "action": best_action
    })

df4 = pd.DataFrame(rows_p4)
df4.head(5)

,state,action
0,1031,-1
1,1152,-1
2,1287,-1
3,1414,-1
4,1413,-1


In [15]:
df4.to_csv("policy-greedy.tsv", sep='\t', index=False)

Submit "policy-greedy.tsv" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "policy-optimal.tsv" with columns state and action.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [16]:
# YOUR CHANGES HERE
epsilon = 0.25
Q = {}

def epsilon_greedy_policy(state, actions):
    if random.random() < epsilon:
        return random.choice(actions)
    if state in Q:
        return max(Q[state], key=Q[state].get)
    return random.choice(actions)

#train longer
for episode in range(500):  
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):

        if curr_state not in Q:
            Q[curr_state] = {a: 0 for a in simulator.get_actions(curr_state)}

        if next_state not in Q:
            Q[next_state] = {a: 0 for a in simulator.get_actions(next_state)}

        #Q-learning update
        max_next_Q = max(Q[next_state].values())
        Q[curr_state][curr_action] = next_reward + 0.9 * max_next_Q

        #stop after reward
        if next_reward > 0:
            break

In [17]:
#extract policy
rows_p5 = []

for state in Q:
    best_action = None
    best_value = float('-inf')

    for action in Q[state]:
        reward, next_state = simulator.get_next_reward_state(state, action)

        #only consider actions leading to known states
        if next_state in Q:
            if Q[state][action] > best_value:
                best_value = Q[state][action]
                best_action = action

    #fallback
    if best_action is None:
        best_action = max(Q[state], key=Q[state].get)

    rows_p5.append({
        "state": state,
        "action": best_action
    })

df5 = pd.DataFrame(rows_p5)
df5.head(5)

,state,action
0,1987,-1
1,1868,1
2,1739,-1
3,1620,-1
4,1491,-1


In [18]:
df5.to_csv("policy-optimal.tsv", sep="\t", index=False)

Submit "policy-optimal.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [19]:
file_content = "none"
with open('acknowledgments.txt', 'w') as f:
    f.write(file_content)